<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 4: LangGraph Advanced — Memory, Async, and Human-in-the-Loop

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Add memory with MemorySaver** — persist conversation state across invocations using checkpointing and `thread_id`
2. **Use async LangGraph** — understand sync vs async and when to use `ainvoke()`
3. **Add human-in-the-loop** — pause a graph for human approval before executing an action

## 1. Setup

In [ ]:
!pip install -q langgraph langchain-openai langchain-community nest-asyncio

In [ ]:
import os
from getpass import getpass

api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
MODEL = "gpt-4o-mini"

---

## 2. Memory with MemorySaver

Try asking a chatbot "What's my name?" after telling it your name — it won't remember. Each `graph.invoke()` is a **super-step** with no memory of previous calls.

**Checkpointing** solves this by saving state between invocations.

| Without Memory | With Memory |
|---|---|
| Each invoke starts fresh | State persists across invokes |
| No conversation history | Full conversation history |
| Stateless | Identified by `thread_id` |

The simplest checkpointer is `MemorySaver` — it stores state in RAM. Let's build a chatbot with tools and memory.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from IPython.display import Image, display

llm = ChatOpenAI(model=MODEL)

class ToolState(TypedDict):
    messages: Annotated[list, add_messages]

# Simple tools
def search(query: str) -> str:
    """Mock search function."""
    responses = {
        "capital": "Paris is the capital of France.",
        "weather": "The weather is sunny with temperatures around 25°C.",
    }
    for key, response in responses.items():
        if key in query.lower():
            return response
    return f"Search results for: {query}"

def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        return f"Result: {eval(expression)}"
    except Exception as e:
        return f"Error: {str(e)}"

tools = [
    Tool(name="search", func=search, description="Useful for searching information"),
    Tool(name="calculator", func=calculate, description="Useful for math calculations"),
]

llm_with_tools = ChatOpenAI(model=MODEL).bind_tools(tools)

def chatbot(state: ToolState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# Build graph with MemorySaver checkpointer
memory = MemorySaver()

graph_builder = StateGraph(ToolState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory_graph = graph_builder.compile(checkpointer=memory)
display(Image(memory_graph.get_graph().draw_mermaid_png()))

In [ ]:
# thread_id identifies a conversation session
config = {"configurable": {"thread_id": "1"}}

# First message
result = memory_graph.invoke({"messages": [{"role": "user", "content": "Hi! My name is Ravi."}]}, config=config)
print(result["messages"][-1].content)

In [ ]:
# Second message — same thread_id, so it remembers!
result = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=config)
print(result["messages"][-1].content)

### Inspecting State

You can inspect the current state and full history of any thread:

In [ ]:
# Current state snapshot
memory_graph.get_state(config)

In [ ]:
# Full state history (most recent first)
list(memory_graph.get_state_history(config))

### Different Threads = Different Conversations

Using a different `thread_id` starts a fresh conversation:

In [ ]:
# Different thread — no memory of Ravi
config2 = {"configurable": {"thread_id": "2"}}
result = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=config2)
print(result["messages"][-1].content)

> **Key insight:** `MemorySaver` stores state in RAM. Pass `checkpointer=memory` when compiling and use `thread_id` in config to maintain separate conversations. Each thread has its own history.

---

## 3. Async LangGraph

In the previous section, we used synchronous calls — `graph.invoke()` blocks until the result is ready. **Async** calls let the program do other work while waiting.

| Sync | Async |
|---|---|
| `graph.invoke(state)` | `await graph.ainvoke(state)` |
| `llm.invoke(messages)` | `await llm.ainvoke(messages)` |
| Blocks until complete | Non-blocking, can run concurrently |
| Fine for simple agents | Better for web requests, I/O-heavy tasks |

**Why async matters:** When your agent calls external APIs, waits for database queries, or handles multiple users, async prevents one slow call from blocking everything else.

`nest_asyncio` allows nested event loops, which is needed to run async code in Jupyter notebooks.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

### Async Chatbot

Let's take the simple chatbot from above and make it async. The only changes are:
- Node function uses `async def` and `await llm.ainvoke()`
- We call `await graph.ainvoke()` instead of `graph.invoke()`

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model=MODEL)

# async node — uses 'async def' and 'await'
async def chatbot_node(state: ChatState):
    response = await llm.ainvoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

async_graph = builder.compile()
display(Image(async_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Use ainvoke instead of invoke
result = await async_graph.ainvoke({"messages": [{"role": "user", "content": "What is the capital of India?"}]})
print(result["messages"][-1].content)

> **Key insight:** Converting to async is straightforward — add `async`/`await` to node functions and use `ainvoke()` instead of `invoke()`. The graph structure stays exactly the same.

---

## 4. Human-in-the-Loop

Agents sometimes need **human approval** before acting. Examples:
- Sending an email on your behalf
- Making a purchase or financial transaction
- Executing a potentially dangerous command
- Deleting or modifying important data

LangGraph supports this with `interrupt_before` — it **pauses the graph** before a specified node runs, letting a human review the pending action and decide whether to approve it.

**How it works:**
1. Compile with `interrupt_before=["node_name"]` + a checkpointer (required to save state while paused)
2. First `invoke()` runs until it hits the interrupt point, then pauses
3. Inspect the pending state with `graph.get_state(config)`
4. Resume with `graph.invoke(None, config)` to approve and continue

### Example: Approve Before Executing

A simple 2-node graph:
- **planner** — LLM decides an action (e.g., "send email to X")
- **executor** — executes the action (prints it)

The graph pauses before `executor` so you can review the plan.

```
START → planner → [INTERRUPT] → executor → END
```

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

class ApprovalState(TypedDict):
    messages: Annotated[list, add_messages]
    action: str

llm_planner = ChatOpenAI(model=MODEL)

def planner(state: ApprovalState):
    """LLM decides what action to take."""
    response = llm_planner.invoke(
        state["messages"] + [{"role": "system", "content": 
            "Based on the user's request, decide on a single action to take. "
            "Respond with ONLY the action description, e.g., 'Send email to boss@company.com with subject: Meeting update'"
        }]
    )
    action = response.content
    return {
        "messages": [response],
        "action": action
    }

def executor(state: ApprovalState):
    """Execute the approved action."""
    action = state["action"]
    result = f"Action executed: {action}"
    print(f"\n>>> EXECUTING: {action}")
    return {"messages": [{"role": "assistant", "content": result}]}

In [ ]:
builder = StateGraph(ApprovalState)
builder.add_node("planner", planner)
builder.add_node("executor", executor)
builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")
builder.add_edge("executor", END)

# interrupt_before pauses the graph BEFORE the executor node runs
# A checkpointer is required — it saves state while the graph is paused
memory = MemorySaver()
approval_graph = builder.compile(checkpointer=memory, interrupt_before=["executor"])
display(Image(approval_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Step 1: Invoke — the graph runs the planner, then PAUSES before executor
config = {"configurable": {"thread_id": "approval-demo"}}

result = approval_graph.invoke(
    {"messages": [{"role": "user", "content": "Send an email to my boss about tomorrow's meeting being postponed"}]},
    config=config
)

print("Graph paused! The planner decided on this action:")
print(f"  → {result['action']}")

In [ ]:
# Step 2: Inspect the pending state
state = approval_graph.get_state(config)
print(f"Next node to run: {state.next}")
print(f"Pending action:   {state.values['action']}")

In [ ]:
# Step 3: Resume — approve by invoking with None (continues from where it paused)
result = approval_graph.invoke(None, config=config)
print(f"\nFinal message: {result['messages'][-1].content}")

> **Key insight:** `interrupt_before` / `interrupt_after` add approval gates to any node. The graph pauses, you inspect the state, and resume when ready. A checkpointer is always required for interrupts.

---

## 5. Key Takeaways

### Quick Reference

| Concept | Code | Purpose |
|---|---|---|
| **Memory** | `compile(checkpointer=MemorySaver())` | Persist state between invocations |
| **Thread ID** | `{"configurable": {"thread_id": "1"}}` | Identify conversation sessions |
| **Async node** | `async def node(state):` | Non-blocking node function |
| **Async invoke** | `await graph.ainvoke(state)` | Non-blocking graph execution |
| **Async LLM call** | `await llm.ainvoke(messages)` | Non-blocking LLM call |
| **Nested async** | `nest_asyncio.apply()` | Enable async in Jupyter notebooks |
| **Human approval** | `interrupt_before=["node"]` | Pause graph before a node |
| **Inspect pause** | `graph.get_state(config)` | See what's pending |
| **Resume** | `graph.invoke(None, config)` | Continue after approval |

### Concept Map

```
NB01 Foundations                    NB02 Advanced
─────────────────                  ─────────────────
State, Nodes, Edges          ───→  Memory (MemorySaver, thread_id)
Conditional Routing                Async (ainvoke, async def)
Tools (bind_tools, ToolNode)       Human-in-the-Loop (interrupt_before)
```

---

## 6. Exercises

### Exercise 1: Custom Tool Agent with Memory (Beginner)
Add a new custom tool (e.g., `weather_lookup` or `dictionary`) to the tools list. Build a graph with tools + memory (`MemorySaver`), and test that:
- The LLM correctly chooses which tool to call
- The conversation memory persists across invocations (tell it your name, then ask for it back)

### Exercise 2: Async Tool Agent (Intermediate)
Convert the tool agent from NB01 (with search and calculator tools) to async:
- Change the chatbot node to `async def` using `await llm.ainvoke()`
- Use `await graph.ainvoke()` instead of `graph.invoke()`
- Verify that tools still work correctly

### Exercise 3: Human-Approved Tool Calls (Intermediate)
Add `interrupt_before=["tools"]` to the tool agent from NB01 so the user must approve each tool call before execution:
- Compile the tool graph with a checkpointer and `interrupt_before=["tools"]`
- Send a message that triggers a tool call
- Inspect the pending state to see which tool will be called
- Resume to execute the tool

---

**Unit 4 Complete!** You've learned LangGraph from fundamentals to advanced patterns: state management, nodes and edges, conditional routing, tools, memory, async execution, and human-in-the-loop approval.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---